<a href="https://colab.research.google.com/github/bhaktii1802/vision-deployment-pipeline/blob/main/02_fair_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Fri Nov 27 04:29:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install onnx onnxruntime-gpu onnxscript tensorrt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 93.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 MB 19.7 MB/s  0:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 20.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.8 MB/s  0:00:00
  Created wheel for tensorrt: filename=tensorrt-11.5.0.63-py3-none-any.whl size=16549 sha256=e7b00972c2123d91efb9f817d554f3af04690fe30472b2176393d7585c9c901e
  Stored in directory: /root/.cache/pip/wheels/91/cc/4b/9a80e49e0067a15f150d02f9f6515d9f0f11df27abb8ae8547
  Created wheel for tensor

In [3]:
import torch, torchvision, os , time

In [4]:
device = "cuda"
model = torchvision.models.resnet18(weights = "DEFAULT").eval().to(device)
dummy = torch.randn(1,3,224,224).to(device)

torch.onnx.export(
    model, dummy, "resnet18.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

print("exported, size MB:", os.path.getsize("resnet18.onnx") / 1e6)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 139MB/s]
/tmp/ipykernel_1048/3460072001.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


exported, size MB: 46.748561


In [6]:
import onnxruntime as ort
import numpy as np
sess = ort.InferenceSession("resnet18.onnx", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
print(sess.get_providers())

x_np = dummy.cpu().numpy()
onnx_out = sess.run(None, {"input": x_np})[0]
with torch.inference_mode():
    torch_out = model(dummy).cpu().numpy()
print("ONNX max diff:", np.abs(onnx_out - torch_out).max())

['CUDAExecutionProvider', 'CPUExecutionProvider']
ONNX max diff: 3.1590462e-06


In [7]:
import tensorrt as trt

logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)
network = builder.create_network()
parser = trt.OnnxParser(network, logger)

with open("resnet18.onnx", "rb") as f:
    ok = parser.parse(f.read())
if not ok:
    for i in range(parser.num_errors):
        print(parser.get_error(i))
    raise SystemExit("ONNX parse failed")

config = builder.create_builder_config()
profile = builder.create_optimization_profile()
profile.set_shape("input", (1, 3, 224, 224), (1, 3, 224, 224), (8, 3, 224, 224))
config.add_optimization_profile(profile)

serialized = builder.build_serialized_network(network, config)
with open("resnet18_fp32.engine", "wb") as f:
    f.write(serialized)
print("engine size MB:", os.path.getsize("resnet18_fp32.engine") / 1e6)

engine size MB: 76.25294


In [8]:
runtime = trt.Runtime(logger)
engine = runtime.deserialize_cuda_engine(serialized)
context = engine.create_execution_context()

x = dummy.contiguous()
y = torch.empty(1, 1000, device="cuda")

context.set_input_shape("input", tuple(x.shape))
context.set_tensor_address("input", x.data_ptr())
context.set_tensor_address("output", y.data_ptr())
stream = torch.cuda.current_stream().cuda_stream

context.execute_async_v3(stream)
torch.cuda.synchronize()
with torch.inference_mode():
    ref = model(x)
print("TensorRT max diff:", (y - ref).abs().max().item())

TensorRT max diff: 4.291534423828125e-06


In [9]:
def bench_torch(n=100):
    with torch.inference_mode():
        for _ in range(20): model(dummy)
        torch.cuda.synchronize()
        t = time.time()
        for _ in range(n): model(dummy)
        torch.cuda.synchronize()
    return (time.time() - t) / n * 1000

def bench_onnx(n=100):
    for _ in range(20): sess.run(None, {"input": x_np})
    t = time.time()
    for _ in range(n): sess.run(None, {"input": x_np})
    return (time.time() - t) / n * 1000

def bench_trt(n=100):
    for _ in range(20): context.execute_async_v3(stream)
    torch.cuda.synchronize()
    t = time.time()
    for _ in range(n): context.execute_async_v3(stream)
    torch.cuda.synchronize()
    return (time.time() - t) / n * 1000

res = {"PyTorch": [], "ONNX Runtime": [], "TensorRT FP32": []}
for _ in range(5):                      # interleaved, so GPU conditions are shared
    res["PyTorch"].append(bench_torch())
    res["ONNX Runtime"].append(bench_onnx())
    res["TensorRT FP32"].append(bench_trt())

base = np.mean(res["PyTorch"])
for k, v in res.items():
    print(f"{k:14s} {np.mean(v):6.2f} ± {np.std(v):.2f} ms   speedup {base/np.mean(v):.2f}x")

PyTorch          2.92 ± 1.22 ms   speedup 1.00x
ONNX Runtime     2.20 ± 0.06 ms   speedup 1.33x
TensorRT FP32    1.28 ± 0.01 ms   speedup 2.29x


In [10]:
import copy

model_h = copy.deepcopy(model).half().eval()      # same weights, converted to FP16
dummy_h = dummy.half()

torch.onnx.export(
    model_h, dummy_h, "resnet18_fp16.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)
print("fp16 onnx size MB:", os.path.getsize("resnet18_fp16.onnx") / 1e6)

/tmp/ipykernel_1048/4009755464.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


fp16 onnx size MB: 23.379131


In [11]:
def build_engine(onnx_path, engine_path):
    builder = trt.Builder(logger)
    network = builder.create_network()
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, "rb") as f:
        ok = parser.parse(f.read())
    if not ok:
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise SystemExit("ONNX parse failed")
    config = builder.create_builder_config()
    profile = builder.create_optimization_profile()
    profile.set_shape("input", (1, 3, 224, 224), (1, 3, 224, 224), (8, 3, 224, 224))
    config.add_optimization_profile(profile)
    blob = builder.build_serialized_network(network, config)
    with open(engine_path, "wb") as f:
        f.write(blob)
    return blob

serialized_fp16 = build_engine("resnet18_fp16.onnx", "resnet18_fp16.engine")
print("fp16 engine size MB:", os.path.getsize("resnet18_fp16.engine") / 1e6)

fp16 engine size MB: 23.574572


In [12]:
engine_h = runtime.deserialize_cuda_engine(serialized_fp16)
context_h = engine_h.create_execution_context()

x_h = dummy_h.contiguous()                                   # FP16 input
y_h = torch.empty(1, 1000, dtype=torch.float16, device="cuda")  # FP16 output

context_h.set_input_shape("input", tuple(x_h.shape))
context_h.set_tensor_address("input", x_h.data_ptr())
context_h.set_tensor_address("output", y_h.data_ptr())

context_h.execute_async_v3(stream)
torch.cuda.synchronize()

with torch.inference_mode():
    ref32 = model(dummy)                                     # FP32 reference
print("FP16 max diff vs FP32 PyTorch:", (y_h.float() - ref32).abs().max().item())
print("same top-1 class:", y_h.float().argmax().item() == ref32.argmax().item())

# Slightly bigger check: top-1 agreement over 50 random inputs
match = 0
for _ in range(50):
    xr = torch.randn(1, 3, 224, 224, device="cuda")
    xr_h = xr.half().contiguous()
    context_h.set_tensor_address("input", xr_h.data_ptr())
    context_h.execute_async_v3(stream)
    torch.cuda.synchronize()
    with torch.inference_mode():
        r = model(xr)
    match += int(y_h.float().argmax().item() == r.argmax().item())
print(f"top-1 agreement on 50 random inputs: {match}/50")

FP16 max diff vs FP32 PyTorch: 0.019617557525634766
same top-1 class: True
top-1 agreement on 50 random inputs: 50/50


In [13]:
# Point the FP16 context back at its original input (the loop above changed it)
context_h.set_tensor_address("input", x_h.data_ptr())
context_h.execute_async_v3(stream)
torch.cuda.synchronize()
rel = ((y_h.float() - ref32).abs().max() / ref32.abs().max()).item()
print(f"FP16 relative error (max diff / max output): {rel:.4f}")

def bench_trt_fp16(n=100):
    for _ in range(20): context_h.execute_async_v3(stream)
    torch.cuda.synchronize()
    t = time.time()
    for _ in range(n): context_h.execute_async_v3(stream)
    torch.cuda.synchronize()
    return (time.time() - t) / n * 1000

res = {"PyTorch": [], "ONNX Runtime": [], "TensorRT FP32": [], "TensorRT FP16": []}
for _ in range(5):                      # all four interleaved
    res["PyTorch"].append(bench_torch())
    res["ONNX Runtime"].append(bench_onnx())
    res["TensorRT FP32"].append(bench_trt())
    res["TensorRT FP16"].append(bench_trt_fp16())

base = np.mean(res["PyTorch"])
for k, v in res.items():
    print(f"{k:14s} {np.mean(v):6.2f} ± {np.std(v):.2f} ms   speedup {base/np.mean(v):.2f}x")

FP16 relative error (max diff / max output): 0.0031
PyTorch          3.51 ± 0.94 ms   speedup 1.00x
ONNX Runtime     2.28 ± 0.15 ms   speedup 1.54x
TensorRT FP32    1.29 ± 0.01 ms   speedup 2.71x
TensorRT FP16    0.50 ± 0.00 ms   speedup 7.07x


In [14]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

ds = datasets.Imagenette("data", split="val", size="160px", download=True, transform=tf)
print("images:", len(ds))
print("classes:", ds.classes)

# Imagenette's 10 classes mapped to their ImageNet-1000 indices
to_imagenet = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]

# 1000 random images (seeded, so the test is repeatable)
idx = torch.randperm(len(ds), generator=torch.Generator().manual_seed(0))[:1000].tolist()
loader = DataLoader(Subset(ds, idx), batch_size=1, num_workers=2)

100%|██████████| 99.0M/99.0M [00:03<00:00, 32.1MB/s]


images: 3925
classes: [('tench', 'Tinca tinca'), ('English springer', 'English springer spaniel'), ('cassette player',), ('chain saw', 'chainsaw'), ('church', 'church building'), ('French horn', 'horn'), ('garbage truck', 'dustcart'), ('gas pump', 'gasoline pump', 'petrol pump', 'island dispenser'), ('golf ball',), ('parachute', 'chute')]


In [15]:
ctx32 = engine.create_execution_context()
ctx16 = engine_h.create_execution_context()

b32_in  = torch.empty(1, 3, 224, 224, device="cuda")
b32_out = torch.empty(1, 1000, device="cuda")
b16_in  = torch.empty(1, 3, 224, 224, dtype=torch.float16, device="cuda")
b16_out = torch.empty(1, 1000, dtype=torch.float16, device="cuda")

for c, i, o in [(ctx32, b32_in, b32_out), (ctx16, b16_in, b16_out)]:
    c.set_input_shape("input", (1, 3, 224, 224))
    c.set_tensor_address("input", i.data_ptr())
    c.set_tensor_address("output", o.data_ptr())

correct = {"PyTorch": 0, "TRT FP32": 0, "TRT FP16": 0}
fp16_differs = 0
n = 0

for img, label in loader:
    img = img.cuda()
    true = to_imagenet[label.item()]

    with torch.inference_mode():
        p_ref = model(img).argmax().item()

    b32_in.copy_(img)
    ctx32.execute_async_v3(stream)
    b16_in.copy_(img.half())
    ctx16.execute_async_v3(stream)
    torch.cuda.synchronize()

    p32 = b32_out.argmax().item()
    p16 = b16_out.argmax().item()

    correct["PyTorch"]  += int(p_ref == true)
    correct["TRT FP32"] += int(p32 == true)
    correct["TRT FP16"] += int(p16 == true)
    fp16_differs        += int(p16 != p32)
    n += 1

for k, v in correct.items():
    print(f"{k:9s} accuracy: {v}/{n} = {100*v/n:.1f}%")
print(f"FP16 prediction differs from FP32 on {fp16_differs}/{n} images")

PyTorch   accuracy: 668/1000 = 66.8%
TRT FP32  accuracy: 668/1000 = 66.8%
TRT FP16  accuracy: 667/1000 = 66.7%
FP16 prediction differs from FP32 on 3/1000 images


In [16]:
from collections import Counter
hit, tot = Counter(), Counter()
for img, label in loader:
    with torch.inference_mode():
        p = model(img.cuda()).argmax().item()
    c = label.item()
    tot[c] += 1
    hit[c] += int(p == to_imagenet[c])
for c in sorted(tot):
    print(f"{ds.classes[c][0]:16s} {hit[c]}/{tot[c]} = {100*hit[c]/tot[c]:.0f}%")

tench            78/86 = 91%
English springer 67/105 = 64%
cassette player  55/89 = 62%
chain saw        72/108 = 67%
church           23/92 = 25%
French horn      79/115 = 69%
garbage truck    65/106 = 61%
gas pump         72/106 = 68%
golf ball        76/93 = 82%
parachute        81/100 = 81%


In [17]:
from collections import Counter
conf = Counter()
for img, label in loader:
    if label.item() != 4:            # 4 = church in Imagenette's class order
        continue
    with torch.inference_mode():
        conf[model(img.cuda()).argmax().item()] += 1
print(conf.most_common(5))

[(663, 34), (497, 23), (442, 7), (406, 4), (884, 3)]


In [18]:
import torchvision
cats = torchvision.models.ResNet18_Weights.DEFAULT.meta["categories"]
for i in [663, 497, 442, 406, 884]:
    print(i, cats[i])

663 monastery
497 church
442 bell cote
406 altar
884 vault
